In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Dict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

load_dotenv(dotenv_path=".env")

In [ ]:
class ChatState(TypedDict):

    messages: Annotated[list[BaseMessage],add_messages]

In [ ]:
llm = ChatGroq(
    model="openai/gpt-oss-20b"
)

def chat_node(state: ChatState):
    # Take user query from state
    messages = state["messages"]

    # Send to llm
    response = llm.invoke(messages)

    # Response store state
    return {"messages": [response]}

In [ ]:
checkpointer = MemorySaver()
graph = StateGraph(ChatState)

#add nodes
graph.add_node("chat_node", chat_node)

#add edges
graph.add_edge(START, "chat_node")
graph.add_edge("chat_node", END)

chatbot = graph.compile(checkpointer=checkpointer)

In [ ]:
chatbot

In [ ]:
thread_id = '1'

while True:
    user_message = input("Type here:")
    print(f"User: {user_message}")
    if user_message.strip().lower() in ["exit","quit","bye"]:
        break

    config = { 'configurable': {'thread_id': thread_id}}
    response = chatbot.invoke({"messages": [HumanMessage(content=user_message)]}, config=config)

    print("AI:",response["messages"][-1].content)




In [ ]:
chatbot.get_state(config=config)